# S2 Inter-Rater Agreement — Yes/No Questions

Agreement for the 41 yes/no questions. **SBERT is excluded** — 'yes'/'no' embeddings
are nearly identical regardless of semantic context, making cosine similarity uninformative.
Only **exact match** and **token Jaccard** are used here.

**Three comparisons (inst_blind):**
- **HH** Human × Human
- **MM** Model × Model  
- **HM** Human × Model

**Free-text questions: `10_align_agreement.ipynb`**

In [ ]:
import sys, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from itertools import combinations
from collections import defaultdict
from IPython.display import display

BASE = Path('/home/david/Desktop/yuna/HPA')
sys.path.insert(0, str(BASE / 'analysis'))

from utils.vqa import VQAAnswerMapper, vqa_accuracy
from utils.load_session import load_human_data, clean_answer as _clean_answer
from utils.constants import VARIANT_ORDER, VARIANT_LABELS, VARIANT_COLORS, GROUP_COLORS

mapper = VQAAnswerMapper()
CT_TO_VARIANT = {'question': 'C', 'weaker_object': 'B', 'pronominalized': 'A'}

# ── Load human data ───────────────────────────────────────────────────────────
participants, common_qids, df, _ = load_human_data(BASE, min_answers=348)
n_participants = df['participant'].nunique()

# ── Answer type map ───────────────────────────────────────────────────────────
q_meta_list  = json.load(open(BASE / 'experiment/s2_v4/s4_question.json'))
q_en         = {q['question_id']: q['question_en'] for q in q_meta_list}
q_kr         = {q['question_id']: q['question_kr'] for q in q_meta_list}
answer_type  = {q['question_id']: q['answer_type'] for q in q_meta_list}
yesno_qids   = {qid for qid in common_qids if answer_type.get(qid) == 'yesno'}

print(f'Participants: {n_participants}  |  Yes/No questions: {len(yesno_qids)}')

In [ ]:
# ── Collect individual rater answers (yes/no questions only) ──────────────────
RATER_MODELS = {
    'Qwen3-VL-8B':        ('vlm',        'pretrained',            'Qwen3-VL-8B-Instruct'),
    'LLaVA-1.5-7B':       ('vlm',        'pretrained',            'llava-1.5-7b-hf'),
    'LLaVA-Mistral':      ('vlm',        'pretrained',            'llava-v1.6-mistral-7b-hf'),
    'LLaVA-Vicuna':       ('vlm',        'pretrained',            'llava-v1.6-vicuna-7b-hf'),
    'LLaVA-1.5 (LM)':     ('lm_decoder', 'lm_decoder/pretrained', 'llava-1.5-7b-hf'),
    'LLaVA-Mistral (LM)': ('lm_decoder', 'lm_decoder/pretrained', 'llava-v1.6-mistral-7b-hf'),
    'LLaVA-Vicuna (LM)':  ('lm_decoder', 'lm_decoder/pretrained', 'llava-v1.6-vicuna-7b-hf'),
    'Qwen3-8B':           ('backbone',   'backbone/pretrained',   'Qwen3-8B'),
}

_yn_norm = lambda s: ('yes' if re.sub(r'[^\w]','',str(s).lower().strip()) in ('yes','y')
                      else ('no' if re.sub(r'[^\w]','',str(s).lower().strip()) in ('no','n')
                      else None))

rater_answers = {}   # rater_id → {(qid, var): normalized yn answer}
rater_group   = {}

# Human
for p in participants:
    pid = p['code']
    rater_group[pid]   = 'human'
    rater_answers[pid] = {}
    for a in p['answers']:
        qid = a['question_id']
        if qid not in yesno_qids: continue
        var = a.get('variant', 'C')
        yn  = _yn_norm(a.get('answer_en', ''))
        if yn:
            rater_answers[pid][(qid, var)] = yn

# Models
for label, (gname, tier, mdir) in RATER_MODELS.items():
    path = BASE / 'evaluation/logits' / tier / mdir / 'vqa_1k_control_inst_blind.jsonl'
    if not path.exists(): print(f'  MISSING: {label}'); continue
    rater_group[label]   = gname
    rater_answers[label] = {}
    for line in open(path):
        ex  = json.loads(line)
        qid = int(ex['question_id'])
        if qid not in yesno_qids: continue
        ga  = ex.get('generated_answers', {})
        for ct, var in CT_TO_VARIANT.items():
            yn = _yn_norm(_clean_answer(ga.get(ct, '')))
            if yn:
                rater_answers[label][(qid, var)] = yn

all_raters    = list(rater_answers.keys())
human_raters  = [r for r in all_raters if rater_group[r] == 'human']
model_raters  = [r for r in all_raters if rater_group[r] != 'human']
print(f'Raters: {len(human_raters)} humans + {len(model_raters)} models')

In [ ]:
# ── Build yes/no pairs DataFrame (exact match score: 1 if same, 0 if different) ─
pair_rows = []
for var in VARIANT_ORDER:
    for qid in sorted(yesno_qids):
        q_ans = {r: rater_answers[r].get((qid, var)) for r in all_raters}
        q_ans = {r: a for r, a in q_ans.items() if a}
        if len(q_ans) < 2: continue

        present = list(q_ans.keys())
        for r1, r2 in combinations(present, 2):
            g1, g2 = rater_group[r1], rater_group[r2]
            if g1 == 'human' and g2 == 'human':   ptype = 'HH'
            elif g1 != 'human' and g2 != 'human': ptype = 'MM'
            else:
                ptype = 'HM'
                if g1 != 'human': r1, r2, g1, g2 = r2, r1, g2, g1

            score = 1 if q_ans[r1] == q_ans[r2] else 0
            pair_rows.append({
                'question_id':     qid,
                'question_en':     q_en.get(qid, ''),
                'question_kr':     q_kr.get(qid, ''),
                'variant':         var,
                'pair_type':       ptype,
                'subject_group_1': g1,
                'subject_1':       r1,
                'subject_group_2': g2,
                'subject_2':       r2,
                'answer_1':        q_ans[r1],
                'answer_2':        q_ans[r2],
                'exact_match':     score,
            })

pairs_df = pd.DataFrame(pair_rows)
print(f'Total pairs: {len(pairs_df):,}')
print(pairs_df.groupby('pair_type').size().rename('n_pairs'))

## HH / MM / HM Agreement by Variant

In [ ]:
# ── Three-way bar chart: mean exact match by pair type × variant ──────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_bt  = {'HH': '#333333', 'MM': '#9C27B0', 'HM': '#FF5722'}

# Left: variant C distribution
ax = axes[0]
C_df = pairs_df[pairs_df.variant == 'C']
for i, pt in enumerate(['HH', 'MM', 'HM'], 1):
    vals = C_df[C_df.pair_type == pt]['exact_match']
    ax.bar(i, vals.mean(), color=colors_bt[pt], alpha=0.8, width=0.5)
    ax.errorbar(i, vals.mean(), yerr=vals.sem()*1.96, color='black', capsize=5)
    ax.text(i, vals.mean() + 0.02, f'{vals.mean():.3f}', ha='center', fontsize=10)
    ax.text(i, -0.04, f'n={len(vals):,}', ha='center', fontsize=7, color='#555')
ax.set_xticks([1,2,3])
ax.set_xticklabels(['HH\nhuman–human', 'MM\nmodel–model', 'HM\nhuman–model'])
ax.set_ylabel('Mean exact match rate')
ax.set_ylim(0, 1.0)
ax.set_title(f'Agreement (variant C, yes/no, n={C_df["question_id"].nunique()} qids)')

# Right: trend C→B→A
ax = axes[1]
x = np.arange(len(VARIANT_ORDER))
for pt, color in colors_bt.items():
    means = [pairs_df[(pairs_df.variant==v)&(pairs_df.pair_type==pt)]['exact_match'].mean()
             for v in VARIANT_ORDER]
    sems  = [pairs_df[(pairs_df.variant==v)&(pairs_df.pair_type==pt)]['exact_match'].sem()
             for v in VARIANT_ORDER]
    ax.plot(x, means, 'o-', color=color, label=pt, lw=2, ms=7)
    ax.fill_between(x, np.array(means)-1.96*np.array(sems),
                       np.array(means)+1.96*np.array(sems), color=color, alpha=0.12)
ax.set_xticks(x)
ax.set_xticklabels(['C (original)', 'B (weaker obj)', 'A (pronominalized)'])
ax.set_ylabel('Mean exact match rate')
ax.set_title('Agreement trend C→B→A (yes/no, ±95% CI)')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Inter-Rater Agreement — Yes/No Questions (inst_blind, exact match)', y=1.02)
plt.tight_layout(); plt.show()

# Summary table
tbl = []
for var in VARIANT_ORDER:
    for pt in ['HH','MM','HM']:
        vals = pairs_df[(pairs_df.variant==var)&(pairs_df.pair_type==pt)]['exact_match']
        tbl.append({'Variant': var, 'Pair type': pt, 'n pairs': len(vals),
                    'Mean': round(vals.mean(),3), 'Std': round(vals.std(),3)})
display(pd.DataFrame(tbl).set_index(['Variant','Pair type']))

## Yes/No Consensus — Per Question

For each question, what fraction of participants said 'yes'?
Sorted from most split (≈50%) to strongest consensus (≈0% or ≈100%).

In [ ]:
# ── Per-question consensus (variant C) ───────────────────────────────────────
yn_df = df[(df['question_id'].isin(yesno_qids)) & (df['variant'] == 'C')].copy()
yn_df['yn'] = yn_df['answer_en'].map(_yn_norm)
yn_df = yn_df.dropna(subset=['yn'])

q_yes = (yn_df.groupby('question_id')
         .apply(lambda g: (g['yn'] == 'yes').sum() / len(g))
         .rename('human_yes_frac').reset_index())
q_yes['question_en'] = q_yes['question_id'].map(q_en)
q_yes['n_humans']    = yn_df.groupby('question_id').size().values
q_yes = q_yes.sort_values('human_yes_frac').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(q_yes)*0.35)))
ys = range(len(q_yes))
ax.barh(ys, q_yes['human_yes_frac'],         color='#4CAF50', alpha=0.85, label='Yes')
ax.barh(ys, 1-q_yes['human_yes_frac'], left=q_yes['human_yes_frac'],
        color='#F44336', alpha=0.85, label='No')
ax.axvline(0.5, color='black', lw=1, ls='--', alpha=0.5)
ax.set_yticks(list(ys))
ax.set_yticklabels([f"{r['question_en'][:55]}…" if len(r['question_en'])>55
                    else r['question_en']
                    for _, r in q_yes.iterrows()], fontsize=7)
ax.set_xlabel('Fraction of participants')
ax.set_title(f'Human consensus — yes/no questions (variant C, n={len(q_yes)} qids)\n'
             f'sorted by yes-rate; dashed = 50/50 split')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

consensus = ((q_yes['human_yes_frac']>=0.8)|(q_yes['human_yes_frac']<=0.2)).sum()
split = ((q_yes['human_yes_frac']>0.4)&(q_yes['human_yes_frac']<0.6)).sum()
print(f'Strong consensus (≥80%/≤20%): {consensus}/{len(q_yes)}')
print(f'Close split (40–60%):          {split}/{len(q_yes)}')

In [ ]:
# ── Human majority vote vs model answer: per-model confusion matrix ───────────
human_maj = (yn_df.groupby('question_id')['yn']
             .apply(lambda s: 'yes' if (s=='yes').mean()>0.5 else 'no')
             .rename('human_maj'))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes_flat = axes.flatten()

for ax, (label, (gname, tier, mdir)) in zip(axes_flat, RATER_MODELS.items()):
    model_ans = {qid: rater_answers[label].get((qid,'C'))
                 for qid in yesno_qids
                 if rater_answers.get(label,{}).get((qid,'C'))}
    model_ser = pd.Series(model_ans, name='model_ans')
    merged    = human_maj.to_frame().join(model_ser).dropna()
    if merged.empty: ax.set_visible(False); continue

    mat = pd.crosstab(merged['human_maj'], merged['model_ans'],
                      rownames=['Human majority'], colnames=['Model answer'])
    mat = mat.reindex(index=['yes','no'], columns=['yes','no'], fill_value=0)
    n   = mat.values.sum()

    sns.heatmap(mat, ax=ax, annot=True, fmt='d', cmap='Blues',
                cbar=False, linewidths=0.5)
    agree = (mat.loc['yes','yes'] + mat.loc['no','no']) / n
    ax.set_title(f'{label}\n(agree={agree:.2f}, n={n})', fontsize=8)
    ax.set_xlabel('Model', fontsize=8); ax.set_ylabel('Human', fontsize=8)
    ax.tick_params(labelsize=8)

plt.suptitle('Human majority vote vs Model answer — yes/no (variant C, inst_blind)', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── Export yes/no pairs table ─────────────────────────────────────────────────
OUT = BASE / 'analysis/session2/exports'
OUT.mkdir(exist_ok=True)
out_path = OUT / 'answer_pairs_yesno.csv'
pairs_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({len(pairs_df):,} rows)')

# Display most split HM pairs (variant C)
hm = pairs_df[(pairs_df.variant=='C')&(pairs_df.pair_type=='HM')].copy()
print('\n── Most disagreed HM pairs (variant C, exact_match=0) ──')
cols = ['question_en','subject_1','subject_2','answer_1','answer_2']
display(hm[hm.exact_match==0].head(10)[cols].reset_index(drop=True))

print('\n── Most agreed HM pairs (variant C, exact_match=1) ──')
display(hm[hm.exact_match==1].head(10)[cols].reset_index(drop=True))